# Phase 1 — Building the Golden Dataset
### Collections Analytics Challenge

**Goal:** turn 17 messy raw tables into a trustworthy analytical layer, and prove the business impact of every cleaning decision.



## 0. Setup

We **read** the raw CSVs and **write** golden tables to a separate folder. The raw files are never modified — that keeps the work reproducible and preserves the "dirty" original as evidence for the ₹27 Cr correction.

In [1]:
import pandas as pd, numpy as np, warnings, os
warnings.filterwarnings('ignore')



import os
import pandas as pd

# ---- Local paths (your machine) ----
RAW = r'C:\Users\Dell\Downloads\collections_30k_dataset (1)'   # read-only source of truth
OUT = r'C:\Users\Dell\Downloads\collections_30k_dataset (1)\golden_output'  # clean copy we build
BIZ_TZ = 'Asia/Kolkata'   # business timezone everything normalises to

os.makedirs(OUT, exist_ok=True)

# ---- Audit trail ----
ledger = []
def log(table, raw, golden, rejected=0, corrected=0, note=''):
    ledger.append(dict(table=table, raw=raw, rejected=rejected,
                        corrected=corrected, golden=golden, note=note))

print('Setup ready. Reading from', RAW)

# ---- Load all CSVs from the folder into a dict of DataFrames ----
csv_files = [f for f in os.listdir(RAW) if f.lower().endswith('.csv')]

dfs = {}
for fname in csv_files:
    fpath = os.path.join(RAW, fname)
    table_name = os.path.splitext(fname)[0]
    df = pd.read_csv(fpath, low_memory=False)
    dfs[table_name] = df
    log(table_name, raw=len(df), golden=len(df), note='loaded')

# ---- Quick sanity check ----
for name, df in dfs.items():
    print(f'{name:30s} shape={df.shape}')

# ledger so far
ledger_df = pd.DataFrame(ledger)
print(ledger_df)

Setup ready. Reading from C:\Users\Dell\Downloads\collections_30k_dataset (1)
accounts                       shape=(30000, 11)
account_status_history         shape=(60000, 8)
agents                         shape=(30000, 8)
agent_sessions                 shape=(15000, 7)
borrowers                      shape=(30600, 8)
calls                          shape=(91350, 11)
call_attempts                  shape=(120000, 9)
call_dispositions              shape=(35000, 8)
campaigns                      shape=(120, 7)
complaints                     shape=(8000, 9)
daily_targeting                shape=(45000, 7)
data_dictionary                shape=(143, 3)
field_visits                   shape=(25000, 10)
payments                       shape=(25500, 9)
promises_to_pay                shape=(18000, 9)
sms_events                     shape=(45000, 8)
vendor_telephony               shape=(15, 6)
whatsapp_events                shape=(60600, 8)
                     table     raw  rejected  corrected  golde

## 1. Reusable cleaning functions

Two operations recur across many tables, so we write them **once** and reuse them. This is the difference between a maintainable pipeline and 17 copies of the same logic.

- **`exact_dedup`** — removes byte-identical rows (proven injected noise). Safe because every column matches.
- **`to_ist`** — converts a timestamp column from its *local wall-clock* zone to Asia/Kolkata. Critical: the data mixes UTC / Kolkata / Dubai, so "3pm" means three different real moments until normalised. Without this, "best calling hour" and monthly trends are wrong by up to 5.5 hours.

In [2]:
def exact_dedup(df):
    """DELETE byte-identical rows (proven injected noise). Returns (clean_df, n_removed)."""
    before = len(df)
    df2 = df.drop_duplicates()
    return df2, before - len(df2)

def to_ist(series, tz_series):
    """Convert local wall-clock timestamps to Asia/Kolkata.
       tz_series gives each row's own timezone; we localise then convert,
       returning tz-naive IST for clean grouping downstream."""
    s = pd.to_datetime(series, errors='coerce')
    out = pd.Series(pd.NaT, index=s.index)
    for tz in tz_series.dropna().unique():
        m = tz_series == tz
        try:
            out[m] = (s[m].dt.tz_localize(tz, ambiguous='NaT', nonexistent='NaT')
                          .dt.tz_convert(BIZ_TZ).dt.tz_localize(None))
        except Exception:
            out[m] = s[m]
    return out

print('Reusable ops defined: exact_dedup(), to_ist()')

Reusable ops defined: exact_dedup(), to_ist()


## 2. Dimension tables (the "who and what")

These describe entities. Two of them — **agents** and **borrowers** — have *scrambled identities* and need entity resolution. The others are lookups.

### 2.1 `vendor_telephony` — telephony vendor lookup
15 vendors, tiny but referenced everywhere. 9 of 15 are INACTIVE — worth flagging for the vendor-performance question later. Just dedup and keep.

In [3]:
v = pd.read_csv(f'{RAW}/vendor_telephony.csv'); vraw=len(v)
v, d = exact_dedup(v)
v.to_parquet(f'{OUT}/dim_vendor.parquet')
log('vendor_telephony', vraw, len(v), rejected=d, note='lookup; 9/15 INACTIVE flagged')
print(f'vendor_telephony: {vraw} -> {len(v)}')

vendor_telephony: 15 -> 15


### 2.2 `accounts` 

`account_id` is the one genuinely clean key in the dataset, so **everything joins to this table**. But its content has three problems we handle without deleting a single row:

1. **Broken borrower links** — 455 null + 2,458 orphan borrower_ids (~10% of accounts). We **tag** them `valid/missing/orphan`. *Dropping them would be the denominator trap.*
2. **`status` lies** — 7,382 "PAID" accounts still owe money. We rename it `status_snapshot` and get real status from the history table instead.
3. **Financial contradictions** — 43% have `outstanding > principal`. We **flag**, we don't "fix" (fixing = fabricating).

This table also becomes the **timezone dictionary**: child event tables with no timezone of their own inherit it via `account_id`.

In [4]:
!pip install pyarrow

a = pd.read_csv(f'{RAW}/accounts.csv'); araw=len(a)
b_ids = set(pd.read_csv(f'{RAW}/borrowers.csv').borrower_id)

# TAG borrower link -- do NOT drop
a['borrower_link'] = np.where(a.borrower_id.isna(), 'missing',
                       np.where(a.borrower_id.isin(b_ids), 'valid', 'orphan'))
# RENAME status -> status_snapshot (point-in-time truth comes from history)
a = a.rename(columns={'status':'status_snapshot'})
# FLAG financial contradiction (do not fix)
a['flag_outstanding_gt_principal'] = a.outstanding_amount > a.principal_amount
a['opened_at_ist'] = to_ist(a.opened_at, a.timezone)

tag_counts = a.borrower_link.value_counts().to_dict()
a.to_parquet(f'{OUT}/dim_account.parquet')
log('accounts', araw, len(a), corrected=len(a),
    note=f"tagged borrower_link {tag_counts}; status->snapshot; 0 dropped")

ACC_TZ = a.set_index('account_id').timezone   # << TIMEZONE DICTIONARY
print('borrower_link tags:', tag_counts)
print('accounts kept:', len(a), '(0 dropped)')


[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


borrower_link tags: {'valid': 27087, 'orphan': 2458, 'missing': 455}
accounts kept: 30000 (0 dropped)


### 2.3 `agents`
30,000 rows collapse to **1,000 real agents**, and every agent_id carries ~30 *conflicting* names/teams/vendors — the identity is genuinely scrambled.

**Survivorship rule (a documented judgment call, not a data-forced fact):** keep the attributes from each agent's **latest `updated_at`** row ("most recent = current truth"), but compute **tenure from the earliest `joined_at`**. We stamp `identity_confidence = LOW` so every downstream agent conclusion is honestly labelled as low-confidence.

In [10]:
ag = pd.read_csv(f'{RAW}/agents.csv'); agraw=len(ag)
ag['joined_at']=pd.to_datetime(ag.joined_at,errors='coerce')
ag['updated_at']=pd.to_datetime(ag.updated_at,errors='coerce')

tenure = ag.groupby('agent_id').joined_at.min().rename('tenure_start')       # earliest
golden_ag = (ag.sort_values('updated_at').groupby('agent_id').tail(1)         # latest attrs
               .drop(columns=['joined_at']).merge(tenure, on='agent_id'))
golden_ag['identity_confidence']='LOW'   # honest flag
golden_ag.to_parquet(f'{OUT}/dim_agent.parquet')
log('agents', agraw, len(golden_ag), rejected=agraw-len(golden_ag),
    note='resolved 30k->1k by latest updated_at; tenure=earliest joined_at; conf=LOW')
print(f'agents: {agraw} -- {len(golden_ag)} (resolved to true agents)')

agents: 30000 -- 1000 (resolved to true agents)


### 2.4 `borrowers` — resolve + repair
600 exact duplicates (delete), then resolve 30,000 → **11,015** borrower_ids by latest `updated_at`. Also cast `phone` from float (`9037419103.0`) to clean string. Contact nulls (phone/email) are **left as-is and tagged** — we never invent contact details.

In [9]:
br = pd.read_csv(f'{RAW}/borrowers.csv'); brraw=len(br)
br, dexact = exact_dedup(br)                                    # DELETE 600 exact dups
br['updated_at']=pd.to_datetime(br.updated_at,errors='coerce')
br['phone']=br.phone.astype('string').str.replace(r'\.0$','',regex=True)  # REPLACE float->str
golden_br = br.sort_values('updated_at').groupby('borrower_id').tail(1)     # survivor=latest
golden_br['identity_confidence']='MEDIUM'
golden_br.to_parquet(f'{OUT}/dim_borrower.parquet')
log('borrowers', brraw, len(golden_br), rejected=dexact,
    corrected=brraw-dexact-len(golden_br),
    note='drop 600 exact; resolve to 11k by latest updated_at; phone->string')
print(f'borrowers: {brraw} -- {len(golden_br)}')

borrowers: 30600 -- 11015


### 2.5 `campaigns` — trust channel, not name
Only 120 rows, but the "inconsistent campaign definition" trap lives here: a campaign named `DIGITAL_FOLLOWUP` is tagged channel `FIELD`. We mark `channel` as the source of truth and flag the name as decorative. `strategy_version` + `start_at` will power the counterfactual (when did targeting change).

In [8]:
cm = pd.read_csv(f'{RAW}/campaigns.csv'); cmraw=len(cm)
cm['channel_source_of_truth']=cm.channel   # use channel, NOT name
cm['name_channel_conflict']=True
cm.to_parquet(f'{OUT}/dim_campaign.parquet')
log('campaigns', cmraw, len(cm), note='channel=truth; name flagged decorative')
print(f'campaigns: {cmraw}- {len(cm)}')

campaigns: 120- 120


## 3. Fact tables (the "what happened")

Event logs. Most are structurally clean and need **timezone normalisation + code harmonisation**, not row surgery. The exceptions — `payments` and `account_status_history` — carry the real analytical traps.

### 3.1 `agent_sessions` — the productivity denominator
Durations are clean (0.5–10h), so `session_hours` is safe to compute now. Two timezones mean we add IST columns for any time-of-day work. We also flag **overlapping sessions** (same agent logged in twice at once) — these would double-count agent-hours if summed naively.

In [11]:
s = pd.read_csv(f'{RAW}/agent_sessions.csv'); sraw=len(s)
s['login_ist']=to_ist(s.login_at, s.timezone)
s['logout_ist']=to_ist(s.logout_at, s.timezone)
s['session_hours']=(pd.to_datetime(s.logout_at)-pd.to_datetime(s.login_at)).dt.total_seconds()/3600
s=s.sort_values(['agent_id','login_ist'])
s['prev_logout']=s.groupby('agent_id').logout_ist.shift(1)
s['overlap_flag']=s.login_ist < s.prev_logout
s=s.drop(columns='prev_logout')
overlaps=int(s.overlap_flag.sum())
s.to_parquet(f'{OUT}/fact_agent_session.parquet')
log('agent_sessions', sraw, len(s), corrected=overlaps,
    note=f'+session_hours,+IST; flagged {overlaps} overlaps for interval-merge')
print(f'agent_sessions: {sraw} rows; {overlaps} overlaps flagged')

agent_sessions: 15000 rows; 222 overlaps flagged


### 3.2 `calls` — dedup + timezone + automated tagging
Delete 1,271 exact dups. Normalise the 3-way timezone (the top fix for "best calling hour"). Tag the 1,827 null-agent calls as **automated** (IVR/dialer) so they don't get attributed to a human. And only trust `duration_sec` for ANSWERED calls — failed calls wrongly show long durations.

In [ ]:
c = pd.read_csv(f'{RAW}/calls.csv'); craw=len(c)
c, dcall = exact_dedup(c)                           # DELETE 1,271 exact dups
c['event_ist']=to_ist(c.event_at, c.timezone)
c['is_automated']=c.agent_id.isna()                 # TAG automated
c['duration_valid']=c.call_status=='ANSWERED'
c.to_parquet(f'{OUT}/fact_call.parquet')
log('calls', craw, len(c), rejected=dcall,
    note=f'drop {dcall} exact; +IST; tag automated({int(c.is_automated.sum())}); duration valid only if ANSWERED')
print(f'calls: {craw} -- {len(c)}; automated tagged: {int(c.is_automated.sum())}')

calls: 91350 -> 90079; automated tagged: 1827


### 3.3 `call_attempts` — inherit timezone from account
Structurally clean (120k unique, every call_id links to a real call). It has no timezone column of its own, so it **inherits its account's timezone** via the dictionary we built. Tag the 2,400 null-vendor rows.

In [13]:
ca = pd.read_csv(f'{RAW}/call_attempts.csv'); caraw=len(ca)
ca, _ = exact_dedup(ca)
ca['event_ist']=to_ist(ca.event_at, ca.account_id.map(ACC_TZ))  # inherit acct TZ
ca['vendor_known']=ca.vendor_id.notna()
ca.to_parquet(f'{OUT}/fact_call_attempt.parquet')
log('call_attempts', caraw, len(ca), note=f'+IST(inherited); {int((~ca.vendor_known).sum())} null-vendor tagged')
print(f'call_attempts: {caraw} rows; null-vendor: {int((~ca.vendor_known).sum())}')

call_attempts: 120000 rows; null-vendor: 2400


### 3.4 `call_dispositions`
The "changed disposition codes" trap: `PTP` and `PROMISE_TO_PAY` are the *same outcome* stored two ways. Counting only one **understates PTP rate by ~50%**. We map `PTP → PROMISE_TO_PAY` into a clean `disposition_clean` column before any rate is computed.

In [14]:
cd = pd.read_csv(f'{RAW}/call_dispositions.csv'); cdraw=len(cd)
CODE_MAP={'PTP':'PROMISE_TO_PAY'}                    # REPLACE: unify duplicate codes
cd['disposition_clean']=cd.disposition_code.replace(CODE_MAP)
cd['event_ist']=to_ist(cd.event_at, cd.account_id.map(ACC_TZ))
unified=int((cd.disposition_code=='PTP').sum())
cd.to_parquet(f'{OUT}/fact_disposition.parquet')
log('call_dispositions', cdraw, len(cd), corrected=unified,
    note=f'unified {unified} PTP->PROMISE_TO_PAY (fixes ~50% PTP undercount)')
print(f'call_dispositions: {unified} PTP codes unified')

call_dispositions: 3904 PTP codes unified


### 3.5 `payments` 
This is where the "11%" story lives. We trace the money through every step:

1. Delete exact-duplicate rows.
2. Deduplicate SUCCESS payments by `payment_reference` (a repeated reference = a **retry**, not new money). Null-reference rows can't be deduped, so we keep and tag them.
3. **True recovery = deduped SUCCESS − REVERSED.**

The result: **₹134 Cr naive → ₹107.5 Cr true.** That ₹27 Cr gap is the fake improvement.

In [15]:
p = pd.read_csv(f'{RAW}/payments.csv'); praw=len(p)
naive_recovery = p[p.payment_status=='SUCCESS'].amount.sum()

p, dexact = exact_dedup(p)                           # DELETE exact dups
succ=p[p.payment_status=='SUCCESS']
succ_ded=pd.concat([succ.dropna(subset=['payment_reference']).drop_duplicates('payment_reference'),
                    succ[succ.payment_reference.isna()]])   # keep null-ref, tagged
ref_removed=len(succ)-len(succ_ded)

p_gold=pd.concat([p[p.payment_status!='SUCCESS'], succ_ded]).copy()
p_gold['event_ist']=to_ist(p_gold.event_at, p_gold.account_id.map(ACC_TZ))
p_gold['is_recovery']=p_gold.payment_status=='SUCCESS'
p_gold['null_reference']=p_gold.payment_reference.isna()
p_gold.to_parquet(f'{OUT}/fact_payment.parquet')

gross=succ_ded.amount.sum()
reversed_amt=p[p.payment_status=='REVERSED'].amount.sum()
true_recovery=gross-reversed_amt
log('payments', praw, len(p_gold), rejected=dexact+ref_removed,
    note=f'drop {dexact} exact +{ref_removed} retry-refs; TRUE Rs {true_recovery/1e7:.1f}Cr vs naive Rs {naive_recovery/1e7:.1f}Cr')
print(f'Naive recovery: Rs {naive_recovery/1e7:.1f} Cr')
print(f'True  recovery: Rs {true_recovery/1e7:.1f} Cr')
print(f'Inflation removed: Rs {(naive_recovery-true_recovery)/1e7:.1f} Cr ({100*(naive_recovery-true_recovery)/true_recovery:.1f}%)')

Naive recovery: Rs 134.1 Cr
True  recovery: Rs 107.5 Cr
Inflation removed: Rs 26.7 Cr (24.8%)


### 3.6 `promises_to_pay` 

The `status=KEPT` label is untrustworthy on its own. We build an **independent** `kept_verified` flag: a promise only counts as kept if a real SUCCESS payment lands within 7 days of the promised date. This exposes a second inflated metric.

In [16]:
pt_g=pd.read_csv(f'{RAW}/promises_to_pay.csv'); ptraw=len(pt_g)
pt_g['event_ist']=to_ist(pt_g.event_at, pt_g.account_id.map(ACC_TZ))
pt_g['promised_date']=pd.to_datetime(pt_g.promised_date,errors='coerce')

# cross-check each PTP against real payments on the same account
pay_by_acct=p_gold[p_gold.is_recovery][['account_id','event_ist']]
chk=pt_g[['ptp_id','account_id','promised_date']].merge(pay_by_acct,on='account_id',how='left')
chk['days']=(chk.event_ist-chk.promised_date).abs().dt.total_seconds()/86400
kept_verified=chk.groupby('ptp_id').days.min()<=7
pt_g['kept_verified']=pt_g.ptp_id.map(kept_verified).fillna(False)
pt_g.to_parquet(f'{OUT}/fact_ptp.parquet')

label_kept=int((pt_g.status=='KEPT').sum()); verified=int(pt_g.kept_verified.sum())
log('promises_to_pay', ptraw, len(pt_g), corrected=verified,
    note=f'+kept_verified: label KEPT={label_kept} vs verified={verified}')
print(f'PTP labelled KEPT: {label_kept}  |  independently verified: {verified}')

PTP labelled KEPT: 4489  |  independently verified: 567


### 3.7 `field_visits`, `whatsapp_events`, `sms_events`, `complaints`
The remaining event tables are clean → normalise timezone, drop the 600 WhatsApp dups, and tag minor issues (ad-hoc field visits with no scheduled time). WhatsApp's `PAYMENT_CLICK` is the strongest digital-conversion signal for the ₹10 Cr question; complaints are the guardrail metric.

In [17]:
# FIELD VISITS
fv = pd.read_csv(f'{RAW}/field_visits.csv'); fvraw=len(fv)
fv['event_ist']=to_ist(fv.event_at, fv.account_id.map(ACC_TZ))
fv['adhoc_visit']=fv.scheduled_at.isna()
fv.to_parquet(f'{OUT}/fact_field_visit.parquet')
log('field_visits', fvraw, len(fv), note=f'{int(fv.adhoc_visit.sum())} ad-hoc tagged; GPS valid')

# WHATSAPP
w = pd.read_csv(f'{RAW}/whatsapp_events.csv'); wraw=len(w)
w, dw = exact_dedup(w)                                # DELETE 600 exact dups
w['event_ist']=to_ist(w.event_at, w.account_id.map(ACC_TZ))
w.to_parquet(f'{OUT}/fact_whatsapp.parquet')
log('whatsapp_events', wraw, len(w), rejected=dw, note=f'drop {dw} exact; +IST; funnel to PAYMENT_CLICK')

# SMS
sm = pd.read_csv(f'{RAW}/sms_events.csv'); smraw=len(sm)
sm, _ = exact_dedup(sm)
sm['event_ist']=to_ist(sm.event_at, sm.account_id.map(ACC_TZ))
sm.to_parquet(f'{OUT}/fact_sms.parquet')
log('sms_events', smraw, len(sm), note='+IST; funnel to CLICKED (thin signal)')

# COMPLAINTS
cp = pd.read_csv(f'{RAW}/complaints.csv'); cpraw=len(cp)
cp['event_ist']=to_ist(cp.event_at, cp.account_id.map(ACC_TZ))
cp.to_parquet(f'{OUT}/fact_complaint.parquet')
log('complaints', cpraw, len(cp), note='clean guardrail table; +IST')
print('field_visits, whatsapp, sms, complaints — done')

field_visits, whatsapp, sms, complaints — done


### 3.8 `daily_targeting`
Flag the 160 duplicate account-day rows. The important find: only 23,344 of 30,000 accounts were ever targeted → **6,656 accounts were never targeted.** That untouched group becomes a natural **control group** for the Phase 4 counterfactual.

In [18]:
dt = pd.read_csv(f'{RAW}/daily_targeting.csv'); dtraw=len(dt)
dt['target_date']=pd.to_datetime(dt.target_date,errors='coerce')
dup_ad=int(dt.duplicated(subset=['account_id','target_date']).sum())
dt['dup_account_day']=dt.duplicated(subset=['account_id','target_date'],keep=False)
never_targeted=30000-dt.account_id.nunique()
dt.to_parquet(f'{OUT}/fact_targeting.parquet')
log('daily_targeting', dtraw, len(dt), corrected=dup_ad,
    note=f'{dup_ad} dup acct-day flagged; {never_targeted} accts never targeted (control group)')
print(f'daily_targeting: {dup_ad} dup acct-days flagged; {never_targeted} never-targeted (control group)')

daily_targeting: 160 dup acct-days flagged; 6656 never-targeted (control group)


### 3.9 `account_status_history` 
The survivorship trap. `recorded_at` conflicts with `event_at` (timezone artifact) so we trust **`event_at`** for ordering. We build the **final status** (last event per account) and flag **7,758 accounts that re-open after a terminal state** — proof that `accounts.status` can't be trusted, and the reason we compute status point-in-time.

In [19]:
h = pd.read_csv(f'{RAW}/account_status_history.csv'); hraw=len(h)
h['event_ist']=to_ist(h.event_at, h.account_id.map(ACC_TZ))   # event_at = truth
h=h.sort_values(['account_id','event_ist'])

final_status=(h.groupby('account_id').tail(1)[['account_id','status']]
                .rename(columns={'status':'final_status'}))

TERM={'PAID','CLOSED','WRITEOFF'}
def reopened(g):
    seen=False
    for st in g.status:
        if seen and st not in TERM: return True
        if st in TERM: seen=True
    return False
viol=int(h.groupby('account_id').apply(reopened).sum())

h.to_parquet(f'{OUT}/fact_status_history.parquet')
final_status.to_parquet(f'{OUT}/dim_account_final_status.parquet')
log('account_status_history', hraw, len(h), corrected=viol,
    note=f'event_at=truth; built point-in-time; {viol} terminal-reopen flagged')
print(f'account_status_history: {viol} terminal-reopen violations flagged')

account_status_history: 7758 terminal-reopen violations flagged


## 4. The cleaning ledger — RAW → REJECTED/CORRECTED → GOLDEN

The full audit trail. Every rejected row is proven noise; every correction traces to a measured issue. This table *is* Deliverable 4 (the Data Quality Report) in compact form.

In [21]:
L=pd.DataFrame(ledger)
print('RAW  ->  REJECTED/CORRECTED  ->  GOLDEN')

for _,r in L.iterrows():
    print(f"{r.table:24s} raw={r.raw:>7,} | rej={r.rejected:>6,} corr={r.corrected:>6,} | golden={r.golden:>7,}")
    print(f"{'':26s}-> {r.note}")

print(f"TOTAL raw:    {L.raw.sum():>10,}")
print(f"TOTAL rejected:{L.rejected.sum():>9,}  (proven noise removed)")
print(f"TOTAL golden: {L.golden.sum():>10,}")
L.to_csv(f'{OUT}/_cleaning_ledger.csv', index=False)
L

RAW  ->  REJECTED/CORRECTED  ->  GOLDEN
accounts                 raw= 30,000 | rej=     0 corr=     0 | golden= 30,000
                          -> loaded
account_status_history   raw= 60,000 | rej=     0 corr=     0 | golden= 60,000
                          -> loaded
agents                   raw= 30,000 | rej=     0 corr=     0 | golden= 30,000
                          -> loaded
agent_sessions           raw= 15,000 | rej=     0 corr=     0 | golden= 15,000
                          -> loaded
borrowers                raw= 30,600 | rej=     0 corr=     0 | golden= 30,600
                          -> loaded
calls                    raw= 91,350 | rej=     0 corr=     0 | golden= 91,350
                          -> loaded
call_attempts            raw=120,000 | rej=     0 corr=     0 | golden=120,000
                          -> loaded
call_dispositions        raw= 35,000 | rej=     0 corr=     0 | golden= 35,000
                          -> loaded
campaigns                raw=    120 | r

,table,raw,rejected,corrected,golden,note
0,accounts,30000,0,0,30000,loaded
1,account_status_history,60000,0,0,60000,loaded
2,agents,30000,0,0,30000,loaded
3,agent_sessions,15000,0,0,15000,loaded
4,borrowers,30600,0,0,30600,loaded
5,calls,91350,0,0,91350,loaded
6,call_attempts,120000,0,0,120000,loaded
7,call_dispositions,35000,0,0,35000,loaded
8,campaigns,120,0,0,120,loaded
9,complaints,8000,0,0,8000,loaded


## 5. The headline financial correction

The single most important output of Phase 1.

In [22]:
print(' HEADLINE FINANCIAL CORRECTION ')
print(f"  Naive recovery (reported):  Rs {naive_recovery/1e7:>7.1f} Cr")
print(f"  True  recovery (golden):    Rs {true_recovery/1e7:>7.1f} Cr")
print(f"  Inflation removed:          Rs {(naive_recovery-true_recovery)/1e7:>7.1f} Cr  ({100*(naive_recovery-true_recovery)/true_recovery:.1f}%)")
print()
print(f"Golden tables written to {OUT}/  ({len([f for f in os.listdir(OUT) if f.endswith('.parquet')])} parquet files)")

 HEADLINE FINANCIAL CORRECTION 
  Naive recovery (reported):  Rs   134.1 Cr
  True  recovery (golden):    Rs   107.5 Cr
  Inflation removed:          Rs    26.7 Cr  (24.8%)

Golden tables written to C:\Users\Dell\Downloads\collections_30k_dataset (1)\golden_output/  (18 parquet files)
